# Saludos de Forty5 con la voz de Epachuko (Colab)

Graba la **mitad impar** de las frases de `shared/voz/saludos.json` y sube cada MP3 al repositorio de Sportfy.
La otra mitad la graba Kaggle a la vez (`tools/voz/kaggle-celda3.py`).

**Una sola vez:** en la barra de la izquierda, 🔑 **Secretos** → añade **GH_TOKEN** con tu clave de GitHub (con permiso de escritura en `jonragitt/sportfy`) y activa *Acceso del cuaderno*.

**Cada vez:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* y luego *Entorno de ejecución → Ejecutar todo*. Si Colab se corta, vuelve a ejecutarlo: sigue por donde iba.

In [ ]:
#@title 1. GPU y clave de GitHub
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No hay GPU. Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4.")
print("✅", gpu.splitlines()[0])
from google.colab import userdata
try:
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
    print("🔑 Clave de GitHub lista")
except Exception:
    raise SystemExit("❌ No encuentro el secreto GH_TOKEN: añádelo en 🔑 Secretos y activa el acceso del cuaderno.")

In [ ]:
#@title 2. Instalar el estudio y el motor de voz (unos minutos)
if os.path.isdir("/content/estudio-de-voz/.git"):
    subprocess.run(["git", "-C", "/content/estudio-de-voz", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Jonragitt/estudio-de-voz", "/content/estudio-de-voz"], check=True)
r = subprocess.run("pip install -q chatterbox-tts yt-dlp", shell=True, capture_output=True, text=True)
subprocess.run("pip uninstall -y -q torchvision", shell=True)  # el de Colab rompe el modelo
r2 = subprocess.run(["python", "-c", "from transformers import LlamaModel; import chatterbox.mtl_tts"], capture_output=True, text=True)
if r2.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:], r2.stderr[-3000:])
    raise SystemExit("❌ El motor de voz no se ha instalado bien (mira el error de arriba).")
print("✅ Motor de voz instalado")

In [ ]:
#@title 3. Grabar y subir a GitHub
import sys, json, urllib.request
RAMA = "claude/project-thread-iw4b9m"
frases = json.load(urllib.request.urlopen(f"https://raw.githubusercontent.com/Jonragitt/Sportfy/{RAMA}/shared/voz/saludos.json"))["lecciones"]
mitad = [l["id"] for i, l in enumerate(frases) if i % 2 == 1]
print(f"🎙️ Colab: {len(mitad)} de {len(frases)} frases")
env = dict(os.environ, REPO="jonragitt/sportfy", RAMA=RAMA, TEXTOS="shared/voz/saludos.json", SALIDA="shared/voz",
           VOZ="shared/voz/ref-colab", SOLO=",".join(mitad), EXPRESIVIDAD="0.5", PAUSAS="0.6",
           CARPETA="/content/repo-sportfy")
p = subprocess.Popen([sys.executable, "-u", "kaggle/narrar_repo.py"], cwd="/content/estudio-de-voz",
                     env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linea in p.stdout:
    print(linea, end="")
p.wait()
if p.returncode:
    raise SystemExit(f"❌ La narración se ha parado (código {p.returncode}). Vuelve a ejecutarlo: sigue por donde iba.")